# Quanto Options pricing under CARMA framework using Monte-Carlo method

## Model parameters definition

In [19]:
#--------------------------------------------------------------------- Imports

import numpy as np
import pandas as pd
from scipy.linalg import expm
import matplotlib.pyplot as plt
from scipy.stats import norm, norminvgauss
from datetime import datetime
from dateutil.relativedelta import relativedelta
from scipy.integrate import quad


#------------------------------------------------------------------- Parameters
# All values loaded from data/levy/carma_montecarlo_final_parameters.csv

# CARMA Parameters
b0_X  = 1.0
b1_X  = -1.69107499948807
b2_X  = 0.0856060699524467
a1_X  = 2.7417368516678
a2_X  = 1.85827342924239
a3_X  = 0.161163594914844

sigma_T_state = 0.76963298150931
b0_Y  = 1.0
b1_Y  = 0.584420888567256
a1_Y  = 0.834602554797278
a2_Y  = 0.02473622057743

# Gaussian driver for temperature increments  (nb06 Q_proc exact Lyapunov)
mu_Y    = 0.004626935939183486
sigma_Y = 0.9619333921285353

# NIG driver for idiosyncratic price increments  (raw dL_J, not unit-normalised)
mu_X    = 0.032544431425439106
delta_X = 0.3350081939529935
alpha_X = 0.9821566545385684
beta_X  = -0.09493955549441858

# Coupling and idiosyncratic price scale
lambda_coupling = 0.0017485173776097822
sigma_J         = 0.036050456023551204   # state-space units: sigma_J * std(dL_J) = sigma_X_marginal


#-------------------------------------------------------- Seasonal trend parameters for Y
'''From nb02 calibration — K_day=3, K_year=1, origin 2020-01-01'''

inputs_Y = {
    "intercept": 10.771397,
    "trend":      0.000006,

    "day_cos_1": -2.773628,
    "day_sin_1": -1.343241,
    "day_cos_2":  0.531266,
    "day_sin_2":  0.000657,
    "day_cos_3":  0.138549,
    "day_sin_3":  0.080022,

    "year_cos_1": -8.946115,
    "year_sin_1": -3.027131,
}

inter_Y = {
    (0,0):  1.607345, (0,1): -0.312687,
    (1,0):  0.756860, (1,1): -0.159530,
    (2,0):  0.222841, (2,1): -0.054528,
    (3,0):  0.260909, (3,1):  0.050530,
    (4,0): -0.384971, (4,1):  0.065902,
    (5,0): -0.039503, (5,1): -0.000543,
}

P_DAY = 24
P_YEAR = 365.25 * 24


#---------------------------------------------------------- Matrices for log prices X
'''The YUIMA convention holds'''

# Companion matrix A_X of shape (3, 3)
AX = np.array([[0,         1,          0],
               [0,         0,          1],
               [-a3_X,    -a2_X,   -a1_X]])

# Noise vector B_X of shape (3, 1)  — no sigma: amplitudes go into dLX_t
BX =  np.array([[0.0], [0.0], [1.0]])

# Selection vector c_X of shape (3,)
cX = np.array([b0_X, b1_X, b2_X])

#---------------------------------------------------------- Matrices for Temperature Y
'''The YUIMA convention holds'''

# Companion matrix A_Y of shape (2, 2)
AY = np.array([[    0,        1  ],
               [-a2_Y,      -a1_Y]])

# Noise vector B_Y of shape (2, 1)  — sigma_T IS in BY
BY = np.array([[0.0], [sigma_T_state]])

# Selection vector c_Y of shape (2,)
cY = np.array([b0_Y, b1_Y])

### Functions definition

In [20]:
#---------------------------------------------------- Precomputation of structural matrices

def structural_matrices(params):
    AX, BX = params['AX'], params['BX']
    AY, BY = params['AY'], params['BY']
    T, n_steps, dt = params['T'], params['n_steps'], params['dt']

    eAXdt = expm(AX * dt)
    eAYdt = expm(AY * dt)

    MXX = np.linalg.inv(AX) @ (eAXdt - np.eye(len(AX))) @ BX   # (3, 1)
    MYY = np.linalg.inv(AY) @ (eAYdt - np.eye(len(AY))) @ BY   # (2, 1)  includes sigma_T

    return {'dt'    : dt,
            'eAXdt' : eAXdt,
            'eAYdt' : eAYdt,
            'MXX'   : MXX,
            'MYY'   : MYY}


#----------------------------------------------- Deterministic function for log prices seasonnal part

file = r"../data/deseasonalised/seasonalities.csv"
df = pd.read_csv(file, index_col=0, parse_dates=True)

seasonal_series = df["log_price_seasonal"].copy()
seasonal_series.index = pd.to_datetime(seasonal_series.index, utc=True)

lookup = (pd.DataFrame({"values" : seasonal_series.values,
                        "month"  : seasonal_series.index.month,
                        "dow"    : seasonal_series.index.dayofweek,
                        "hour"   : seasonal_series.index.hour})
          .groupby(["month", "dow", "hour"])["values"]
          .mean()
          .to_dict())

def Lambda_S(params):
    t = params['end_date']
    key = (t.month, t.dayofweek, t.hour)
    return lookup.get(key, np.nan)


#----------------------------------------------- Deterministic function for temperature seasonnal part

def day_features(t):
    return np.array([
        np.cos(2*np.pi*1*t/P_DAY), np.sin(2*np.pi*1*t/P_DAY),
        np.cos(2*np.pi*2*t/P_DAY), np.sin(2*np.pi*2*t/P_DAY),
        np.cos(2*np.pi*3*t/P_DAY), np.sin(2*np.pi*3*t/P_DAY),
    ])

def year_features(t):
    return np.array([
        np.cos(2*np.pi*1*t/P_YEAR),
        np.sin(2*np.pi*1*t/P_YEAR),
    ])

def Lambda_Y(params):
    t = params['O']
    y = inputs_Y["intercept"] + inputs_Y["trend"] * t
    y += inputs_Y["day_cos_1"] * np.cos(2*np.pi*1*t/P_DAY)
    y += inputs_Y["day_sin_1"] * np.sin(2*np.pi*1*t/P_DAY)
    y += inputs_Y["day_cos_2"] * np.cos(2*np.pi*2*t/P_DAY)
    y += inputs_Y["day_sin_2"] * np.sin(2*np.pi*2*t/P_DAY)
    y += inputs_Y["day_cos_3"] * np.cos(2*np.pi*3*t/P_DAY)
    y += inputs_Y["day_sin_3"] * np.sin(2*np.pi*3*t/P_DAY)
    y += inputs_Y["year_cos_1"] * np.cos(2*np.pi*1*t/P_YEAR)
    y += inputs_Y["year_sin_1"] * np.sin(2*np.pi*1*t/P_YEAR)
    d = day_features(t)
    v = year_features(t)
    for (i, j), c in inter_Y.items():
        y += c * d[i] * v[j]
    return y


#--------------------------------------------------------------- Discount factor

def discount_factor(r, t):
    r = r / 8760
    return np.exp(-r * t)


#----------------------------------------------------------- Lévy increments

def sample_gaussian(mu, sigma):
    T, n_steps, dt, N = params['T'], params['n_steps'], params['dt'], params['N']
    return np.random.normal(loc=mu, scale=sigma, size=(int(n_steps), int(N)))


def sample_NIG(alpha, beta, delta, mu):
    T, n_steps, dt, N = params['T'], params['n_steps'], params['dt'], params['N']
    a = alpha * delta   # scipy_a
    b = beta  * delta   # scipy_b
    return norminvgauss.rvs(a, b, loc=mu, scale=delta, size=(int(n_steps), int(N)))


#-------------------------------------------------------------- Coupled paths simulation

def simulate_paths(params, matrices):

    N, n_steps = params['N'], params['n_steps']
    alpha_X, beta_X, delta_X, mu_X = params['alpha_X'], params['beta_X'], params['delta_X'], params['mu_X']
    mu_Y, sigma_Y     = params['mu_Y'], params['sigma_Y']
    lambda_c, sigma_J = params['lambda_coupling'], params['sigma_J']
    sigma_T           = params['sigma_T_state']

    np.random.seed(42)

    dLY = sample_gaussian(mu_Y, sigma_Y)             # (n_steps, N)  temperature driver
    dLJ = sample_NIG(alpha_X, beta_X, delta_X, mu_X) # (n_steps, N)  idiosyncratic price driver

    ZX = np.zeros((N, 3))
    ZY = np.zeros((N, 2))

    eAXdt = matrices['eAXdt']
    eAYdt = matrices['eAYdt']
    MXX   = matrices['MXX']    # inv(AX)(eAX-I) e_X
    MYY   = matrices['MYY']    # inv(AY)(eAY-I) sigma_T e_T  — sigma_T already inside

    for t in range(int(n_steps)):
        dLY_t = dLY[t]   # (N,)
        dLJ_t = dLJ[t]   # (N,)

        # G_joint[pT:, 0] = lambda * sigma_T * e_X  →  coefficient is lambda * sigma_T
        # G_joint[pT:, 1] = sigma_J * e_X
        dLX_t = lambda_c * sigma_T * dLY_t + sigma_J * dLJ_t

        ZX = ZX @ eAXdt.T + np.outer(dLX_t, MXX.flatten())
        ZY = ZY @ eAYdt.T + np.outer(dLY_t, MYY.flatten())

    X_T = ZX @ params['cX']
    Y_T = ZY @ params['cY']

    S_T     = np.exp(Lambda_S(params) + X_T) - 1000
    Y_hat_T = Lambda_Y(params) + Y_T

    return S_T, Y_hat_T


#------------------------------------------------------------------- Monte-Carlo pricer

def Monte_Carlo_price(payoff, params, matrices, confidence=0.95):
    S_T, Y_hat_T = simulate_paths(params, matrices)
    K_S, K_Y = params['K_S'], params['K_Y']
    payoff_distribution = payoff(S_T, K_S, Y_hat_T, K_Y)

    price = discount_factor(params['r'], params['T']) * np.mean(payoff_distribution)
    se    = discount_factor(params['r'], params['T']) * np.std(payoff_distribution) / np.sqrt(params['N'])
    z     = norm.ppf((1 + confidence) / 2)
    ci    = (price - z * se, price + z * se)
    return price, ci, se


#------------------------------------------------------------- Payoff functions

def spot_forward(S_T, K_S, Y_hat_T, K_Y):
    return S_T

def payoff_plain_energy_call(S_T, K_S, Y_hat_T, K_Y):
    return np.maximum(S_T - K_S, 0)

def payoff_plain_energy_put(S_T, K_S, Y_hat_T, K_Y):
    return np.maximum(K_S - S_T, 0)

def payoff_plain_temperature_call(S_T, K_S, Y_hat_T, K_Y):
    return np.maximum(Y_hat_T - K_Y, 0)

def payoff_plain_temperature_put(S_T, K_S, Y_hat_T, K_Y):
    return np.maximum(K_Y - Y_hat_T, 0)

def payoff_temperature_barrier(S_T, K_S, Y_hat_T, K_Y):
    return (Y_hat_T > K_Y).astype(float)

def payoff_quanto_call(S_T, K_S, Y_hat_T, K_Y):
    return np.maximum(S_T - K_S, 0) * (Y_hat_T > K_Y).astype(float)

## Plain energy call pricing

#### $\mathbb{E}[(S_T - K_S)^+]$

#### Monte-Carlo price computation

In [21]:
start_date  = pd.Timestamp("2026-05-13 12:00:00")
end_date    = pd.Timestamp("2026-06-12 19:00:00")
origin_date = pd.Timestamp("2020-01-01 00:00:00")

T = (end_date - start_date).total_seconds() / 3600
O = (end_date - origin_date).total_seconds() / 3600

params = {'AX': AX, 'BX': BX, 'cX': cX,
          'AY': AY, 'BY': BY, 'cY': cY,
          'mu_X': mu_X, 'delta_X': delta_X, 'alpha_X': alpha_X, 'beta_X': beta_X,
          'mu_Y': mu_Y, 'sigma_Y': sigma_Y,
          'lambda_coupling': lambda_coupling, 'sigma_J': sigma_J,
          'sigma_T_state': sigma_T_state,
          'start_date': start_date, 'end_date': end_date, 'origin_date': origin_date,
          'T': T, 'O': O,
          'N': 10_000, 'dt': 1, 'n_steps': T,
          'r': 0, 'K_S': 130, 'K_Y': 10}

matrices = structural_matrices(params)

MC_price, confidence_interval, standard_error = Monte_Carlo_price(payoff_plain_energy_call, params, matrices)
print(f"Monte-Carlo price  : {MC_price:.4f}")
print(f"Std error          : {standard_error:.4f}")
print(f"Confidence interval 95% : [{confidence_interval[0]:.4f}, {confidence_interval[1]:.4f}]")
print(f"Relative error     : {standard_error/MC_price*100:.2f}%")

Monte-Carlo price  : 16.8463
Std error          : 0.2207
Confidence interval 95% : [16.4138, 17.2789]
Relative error     : 1.31%


In [22]:
print(np.exp(Lambda_S(params))-1000)

136.02465319417365


### Call Put parity

##### The following relation should be satisfied : $C - P = \mathbb{E}[S_T] - K_S$

In [23]:
call, _, _ = Monte_Carlo_price(payoff_plain_energy_call, params, matrices)
put,  _, _ = Monte_Carlo_price(payoff_plain_energy_put,  params, matrices)
forward,   _, _ = Monte_Carlo_price(spot_forward, params, matrices)         #E[S_T]

print(f"C - P          : {call - put:.4f}")
print(f"E[S_T] - K_S   : {forward - params['K_S']:.4f}")

C - P          : 6.8348
E[S_T] - K_S   : 6.8348


## Plain temperature call pricing

#### $\mathbb{E}[(Y_T - K_Y)^+]$

In [24]:
start_date  = pd.Timestamp("2026-05-13 12:00:00")
end_date    = pd.Timestamp("2026-06-12 19:00:00")
origin_date = pd.Timestamp("2020-01-01 00:00:00")

T = (end_date - start_date).total_seconds() / 3600
O = (end_date - origin_date).total_seconds() / 3600

params = {'AX': AX, 'BX': BX, 'cX': cX,
          'AY': AY, 'BY': BY, 'cY': cY,
          'mu_X': mu_X, 'delta_X': delta_X, 'alpha_X': alpha_X, 'beta_X': beta_X,
          'mu_Y': mu_Y, 'sigma_Y': sigma_Y,
          'lambda_coupling': lambda_coupling, 'sigma_J': sigma_J,
          'sigma_T_state': sigma_T_state,
          'start_date': start_date, 'end_date': end_date, 'origin_date': origin_date,
          'T': T, 'O': O,
          'N': 10_000, 'dt': 1, 'n_steps': T,
          'r': 0, 'K_S': 130, 'K_Y': 18}

matrices = structural_matrices(params)

MC_price, confidence_interval, standard_error = Monte_Carlo_price(payoff_plain_temperature_call, params, matrices)
print(f"Monte-Carlo price  : {MC_price:.4f}")
print(f"Std error          : {standard_error:.4f}")
print(f"Confidence interval 95% : [{confidence_interval[0]:.4f}, {confidence_interval[1]:.4f}]")
print(f"Relative error     : {standard_error/MC_price*100:.2f}%")

Monte-Carlo price  : 2.1072
Std error          : 0.0250
Confidence interval 95% : [2.0583, 2.1562]
Relative error     : 1.19%


In [25]:
print(Lambda_Y(params))

19.044207741794747


## Barrier option on temperature

### $\mathbb{E}[\mathbb{1}_{Y_T > K_Y}]$

In [26]:
start_date  = pd.Timestamp("2026-05-13 12:00:00")
end_date    = pd.Timestamp("2026-06-12 19:00:00")
origin_date = pd.Timestamp("2020-01-01 00:00:00")

T = (end_date - start_date).total_seconds() / 3600
O = (end_date - origin_date).total_seconds() / 3600

params = {'AX': AX, 'BX': BX, 'cX': cX,
          'AY': AY, 'BY': BY, 'cY': cY,
          'mu_X': mu_X, 'delta_X': delta_X, 'alpha_X': alpha_X, 'beta_X': beta_X,
          'mu_Y': mu_Y, 'sigma_Y': sigma_Y,
          'lambda_coupling': lambda_coupling, 'sigma_J': sigma_J,
          'sigma_T_state': sigma_T_state,
          'start_date': start_date, 'end_date': end_date, 'origin_date': origin_date,
          'T': T, 'O': O,
          'N': 10_000, 'dt': 1, 'n_steps': T,
          'r': 0, 'K_S': 130, 'K_Y': 18}

matrices = structural_matrices(params)

MC_price, confidence_interval, standard_error = Monte_Carlo_price(payoff_temperature_barrier, params, matrices)
print(f"Monte-Carlo price  : {MC_price:.4f}")
print(f"Std error          : {standard_error:.4f}")
print(f"Confidence interval 95% : [{confidence_interval[0]:.4f}, {confidence_interval[1]:.4f}]")
print(f"Relative error     : {standard_error/MC_price*100:.2f}%")

Monte-Carlo price  : 0.6292
Std error          : 0.0048
Confidence interval 95% : [0.6197, 0.6387]
Relative error     : 0.77%


## Quanto option

### $\mathbb{E}[(Y_T - K_Y)^+\mathbb{1}_{Y_T > K_Y}]$

In [27]:
start_date  = pd.Timestamp("2026-05-13 12:00:00")
end_date    = pd.Timestamp("2026-06-12 19:00:00")
origin_date = pd.Timestamp("2020-01-01 00:00:00")

T = (end_date - start_date).total_seconds() / 3600
O = (end_date - origin_date).total_seconds() / 3600

params = {'AX': AX, 'BX': BX, 'cX': cX,
          'AY': AY, 'BY': BY, 'cY': cY,
          'mu_X': mu_X, 'delta_X': delta_X, 'alpha_X': alpha_X, 'beta_X': beta_X,
          'mu_Y': mu_Y, 'sigma_Y': sigma_Y,
          'lambda_coupling': lambda_coupling, 'sigma_J': sigma_J,
          'sigma_T_state': sigma_T_state,
          'start_date': start_date, 'end_date': end_date, 'origin_date': origin_date,
          'T': T, 'O': O,
          'N': 10_000, 'dt': 1, 'n_steps': T,
          'r': 0, 'K_S': 136, 'K_Y': 18}

matrices = structural_matrices(params)

MC_price, confidence_interval, standard_error = Monte_Carlo_price(payoff_quanto_call, params, matrices)
print(f"Monte-Carlo price  : {MC_price:.4f}")
print(f"Std error          : {standard_error:.4f}")
print(f"Confidence interval 95% : [{confidence_interval[0]:.4f}, {confidence_interval[1]:.4f}]")
print(f"Relative error     : {standard_error/MC_price*100:.2f}%")

Monte-Carlo price  : 8.8179
Std error          : 0.1763
Confidence interval 95% : [8.4723, 9.1634]
Relative error     : 2.00%


## Suite : Calculer des prix futures comme moyenne arithmétique des prix spot sur une période (month, quarter, year) et pricer l'option sur future (assimilable à une option asiatique)